# Data cleaning

* Removal of full duplicit rows
* Removal of instances assigned to more categories at the same time
* Merge of perex and title columns into a new combined text column
* Remove the smallest categories
* Split data to train, validation and test splits with concrete seed

### Module imports

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

import sys

In [2]:
sys.path.insert(0, str(Path.cwd().parent))

from src.config import RAW_DATA_PATH, CLEAN_DATA_PATH
from src.config import SEED, VAL_SIZE, TEST_SIZE
from src.config import MIN_CLASS_COUNT

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

In [4]:
original_row_count = df.shape[0]
print(f"We have loaded {original_row_count} rows")

We have loaded 111218 rows


In [5]:
df.head()

,category,rss_title,rss_perex
0,biatlon,"Krčmář dojel v hromadném závodě devátý, díky s...",Závod s hromadným startem v německém Oberhofu ...
1,biatlon,Česká vlajka byla v Pokljuce vidět i ve štafet...,Galerie
2,biatlon,Živě: Stíhací závod biatlonistek v Ruhpoldingu,"15. 1., 14:45"
3,fotbal,Bakoš dostal herdu do nosu a v zápase plném ka...,Slovenský útočník Marek Bakoš zařídil Plzni gó...
4,fotbal,"My moc chtěli a Plzeň moc nechtěla, zní z Ďolí...",Český fotbal hledá viníka skandálu odloženého ...


### Drop duplicate rows

In [6]:
df.drop_duplicates(inplace=True)
no_duplicates_row_count = df.shape[0]
print(f"We have dropped {original_row_count - no_duplicates_row_count} rows")

We have dropped 7185 rows


### Drop conflicting rows

In [7]:
conflict = df.groupby(["rss_title", "rss_perex"])["category"].transform("nunique") > 1
df = df[~conflict]

In [8]:
print(f"We have dropped {no_duplicates_row_count - df.shape[0]} conflicting rows")

We have dropped 27 conflicting rows


### Drop instances associated with categories that have too few samples

In [ ]:
counts = df["category"].value_counts()
small = counts[counts < MIN_CLASS_COUNT].index.tolist()
print(f"The classes that are too small are {small}")

Dropping them

In [10]:
df = df[~df["category"].isin(small)].reset_index(drop=True)

### Build the text column

In [11]:
df["text"] = df["rss_title"] + " " + df["rss_perex"]
df = df.reset_index(drop=True)
df.head(3)

,category,rss_title,rss_perex,text
0,biatlon,"Krčmář dojel v hromadném závodě devátý, díky s...",Závod s hromadným startem v německém Oberhofu ...,"Krčmář dojel v hromadném závodě devátý, díky s..."
1,biatlon,Česká vlajka byla v Pokljuce vidět i ve štafet...,Galerie,Česká vlajka byla v Pokljuce vidět i ve štafet...
2,biatlon,Živě: Stíhací závod biatlonistek v Ruhpoldingu,"15. 1., 14:45",Živě: Stíhací závod biatlonistek v Ruhpoldingu...


# Split to train, val and test splits

Split to train + val AND test (combined)

In [12]:
val_test_size = VAL_SIZE + TEST_SIZE                
train_idx, val_test_idx = train_test_split(
    df.index, test_size=val_test_size, stratify=df["category"], random_state=SEED,
)

Split val AND test set to validation and testing sets that are separate

In [13]:
test_frac = TEST_SIZE / val_test_size           
val_idx, test_idx = train_test_split(
    val_test_idx, test_size=test_frac, stratify=df.loc[val_test_idx, "category"], random_state=SEED,
)

Assign to appropriate columns

In [14]:
df["split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

In [15]:
print(df["split"].value_counts(normalize=True).round(3))

split
train    0.8
test     0.1
val      0.1
Name: proportion, dtype: float64


### Export to intermediary file

In [17]:
df.to_parquet(CLEAN_DATA_PATH)
print(f"Saved: {CLEAN_DATA_PATH}  ({len(df)} rows)")

Saved: /home/jovyan/sport_article_classifier/data/clean.parquet  (104002 rows)
